In [1]:
# Check whether easydiffraction is installed; install it if needed.
# Required for remote environments such as Google Colab.
import importlib.util

if importlib.util.find_spec('easydiffraction') is None:
    %pip install easydiffraction

# Structure Refinement: Co2SiO4, D20

This example demonstrates a Rietveld refinement of Co2SiO4 crystal
structure using constant wavelength neutron powder diffraction data
from D20 at ILL.

It also shows different ways to set free parameters: standard
one-by-one and batch setting.

## Import Library

In [2]:
from easydiffraction import ExperimentFactory
from easydiffraction import Project
from easydiffraction import StructureFactory
from easydiffraction import download_data

## Define Structure

This section shows how to add structures and modify their
parameters.

#### Create Structure

In [3]:
structure = StructureFactory.from_scratch(name='cosio')

#### Set Space Group

In [4]:
structure.space_group.name_h_m = 'P n m a'
structure.space_group.it_coordinate_system_code = 'abc'

#### Set Unit Cell

In [5]:
structure.cell.length_a = 10.3
structure.cell.length_b = 6.0
structure.cell.length_c = 4.8

#### Set Atom Sites

In [6]:
structure.atom_sites.create(
    label='Co1',
    type_symbol='Co',
    fract_x=0,
    fract_y=0,
    fract_z=0,
    wyckoff_letter='a',
    adp_iso=0.5,
)
structure.atom_sites.create(
    label='Co2',
    type_symbol='Co',
    fract_x=0.279,
    fract_y=0.25,
    fract_z=0.985,
    wyckoff_letter='c',
    adp_iso=0.5,
)
structure.atom_sites.create(
    label='Si',
    type_symbol='Si',
    fract_x=0.094,
    fract_y=0.25,
    fract_z=0.429,
    wyckoff_letter='c',
    adp_iso=0.5,
)
structure.atom_sites.create(
    label='O1',
    type_symbol='O',
    fract_x=0.091,
    fract_y=0.25,
    fract_z=0.771,
    wyckoff_letter='c',
    adp_iso=0.5,
)
structure.atom_sites.create(
    label='O2',
    type_symbol='O',
    fract_x=0.448,
    fract_y=0.25,
    fract_z=0.217,
    wyckoff_letter='c',
    adp_iso=0.5,
)
structure.atom_sites.create(
    label='O3',
    type_symbol='O',
    fract_x=0.164,
    fract_y=0.032,
    fract_z=0.28,
    wyckoff_letter='d',
    adp_iso=0.5,
)

## Define Experiment

This section shows how to add experiments, configure their parameters,
and link the structures defined in the previous step.

#### Download Measured Data

In [7]:
data_path = download_data(id=12, destination='data')

Getting data...


Data #12: Co2SiO4, D20 (ILL)


✅ Data #12 downloaded to 'data/ed-12.xye'


#### Create Experiment

In [8]:
expt = ExperimentFactory.from_data_path(name='d20', data_path=data_path)

#### Set Instrument

In [9]:
expt.instrument.setup_wavelength = 1.87
expt.instrument.calib_twotheta_offset = 0.1

#### Set Peak Profile

In [10]:
expt.show_peak_profile_types()

Peak profile types


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,,Type,Description
1,*,pseudo-voigt,CWL pseudo-Voigt profile
2,,pseudo-voigt + empirical asymmetry,CWL pseudo-Voigt profile with empirical asymmetry correction.


In [11]:
expt.peak_profile_type = 'pseudo-voigt + empirical asymmetry'

⚠️ Switching peak profile type discards existing peak parameters.                                                                 


Peak profile type for experiment 'd20' changed to


pseudo-voigt + empirical asymmetry


In [12]:
expt.peak.broad_gauss_u = 0.3
expt.peak.broad_gauss_v = -0.5
expt.peak.broad_gauss_w = 0.4

#### Set Background

In [13]:
expt.show_background_types()

Background types


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,,Type,Description
1,,chebyshev,Chebyshev polynomial background
2,*,line-segment,Linear interpolation between points


In [14]:
expt.background.create(id='1', x=8, y=500)
expt.background.create(id='2', x=9, y=500)
expt.background.create(id='3', x=10, y=500)
expt.background.create(id='4', x=11, y=500)
expt.background.create(id='5', x=12, y=500)
expt.background.create(id='6', x=15, y=500)
expt.background.create(id='7', x=25, y=500)
expt.background.create(id='8', x=30, y=500)
expt.background.create(id='9', x=50, y=500)
expt.background.create(id='10', x=70, y=500)
expt.background.create(id='11', x=90, y=500)
expt.background.create(id='12', x=110, y=500)
expt.background.create(id='13', x=130, y=500)
expt.background.create(id='14', x=150, y=500)

#### Set Linked Phases

In [15]:
expt.linked_phases.create(id='cosio', scale=1.0)

## Define Project

The project object is used to manage the structure, experiment, and
analysis.

#### Create Project

In [16]:
project = Project()

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

#### Add Structure

In [17]:
project.structures.add(structure)

#### Add Experiment

In [18]:
project.experiments.add(expt)

## Perform Analysis

This section shows the analysis process, including how to set up
calculation and fitting engines.

#### Plot Measured vs Calculated

In [19]:
project.display.plotter.plot_meas_vs_calc(expt_name='d20', show_residual=True)

In [20]:
project.display.plotter.plot_meas_vs_calc(expt_name='d20', x_min=41, x_max=54, show_residual=True)

#### Set Free Parameters

In [21]:
structure.cell.length_a.free = True
structure.cell.length_b.free = True
structure.cell.length_c.free = True

for atom_site in structure.atom_sites:
    for parameter in ('fract_x', 'fract_y', 'fract_z'):
        getattr(atom_site, parameter).free = True

for atom_site in structure.atom_sites:
    atom_site.adp_iso.free = True

for label in ('O1', 'O2', 'O3'):
    atom_site = structure.atom_sites[label]
    atom_site.occupancy.free = True

In [22]:
expt.linked_phases['cosio'].scale.free = True

expt.instrument.calib_twotheta_offset.free = True

expt.peak.broad_gauss_u.free = True
expt.peak.broad_gauss_v.free = True
expt.peak.broad_gauss_w.free = True
expt.peak.broad_lorentz_y.free = True

expt.peak.asym_empir_2.free = True

for point in expt.background:
    point.y.free = True

#### Set Constraints

Set aliases for parameters.

In [23]:
project.analysis.aliases.create(
    label='biso_Co1',
    param=project.structures['cosio'].atom_sites['Co1'].adp_iso,
)
project.analysis.aliases.create(
    label='biso_Co2',
    param=project.structures['cosio'].atom_sites['Co2'].adp_iso,
)

Set constraints.

In [24]:
project.analysis.constraints.create(expression='biso_Co2 = biso_Co1')

#### Run Fitting

In [25]:
project.analysis.fit()

Standard fitting


📋 Using experiment 🔬 'd20' for 'single' fitting


🚀 Starting fit process with 'lmfit (leastsq)'...


📈 Goodness-of-fit (reduced χ²) change:


,iteration,χ²,improvement [%]
1,1,426.61,
2,54,74.15,82.6% ↓
3,105,39.61,46.6% ↓
4,157,21.74,45.1% ↓
5,210,14.62,32.7% ↓
6,261,10.95,25.1% ↓
7,312,9.54,12.9% ↓
8,363,8.21,14.0% ↓
9,414,5.66,31.1% ↓
10,465,4.45,21.3% ↓


...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

🏆 Best goodness-of-fit (reduced χ²) is 4.40 at iteration 704


✅ Fitting complete.


In [26]:
project.analysis.display.fit_results()

Fit results


✅ Success: True


⏱️ Fitting time: 35.84 seconds


📏 Goodness-of-fit (reduced χ²): 4.40


📏 R-factor (Rf): 2.99%


...

📏 Weighted R-factor (wR): 4.54%


📈 Fitted parameters:


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,datablock,category,entry,parameter,start,fitted,uncertainty,units,change
1,cosio,cell,,length_a,10.3000,10.3084,N/A,Å,0.08 % ↑
2,cosio,cell,,length_b,6.0000,6.0036,N/A,Å,0.06 % ↑
3,cosio,cell,,length_c,4.8000,4.7864,N/A,Å,0.28 % ↓
4,cosio,atom_site,Co1,fract_x,0.0000,0.0000,N/A,,N/A
5,cosio,atom_site,Co1,fract_y,0.0000,0.0000,N/A,,N/A
6,cosio,atom_site,Co1,fract_z,0.0000,0.0000,N/A,,N/A
7,cosio,atom_site,Co1,adp_iso,0.5000,0.6716,N/A,Å²,34.32 % ↑
8,cosio,atom_site,Co2,fract_x,0.2790,0.2792,N/A,,0.06 % ↑
9,cosio,atom_site,Co2,fract_y,0.2500,0.2500,N/A,,0.00 % ↓
10,cosio,atom_site,Co2,fract_z,0.9850,0.9850,N/A,,0.00 % ↓


In [27]:
project.display.plotter.plot_param_correlations()

⚠️ Correlation matrix is unavailable for this fit. Use the lmfit minimizer and ensure covariance estimation succeeds.             


#### Plot Measured vs Calculated

In [28]:
project.display.plotter.plot_meas_vs_calc(expt_name='d20', show_residual=True)

In [29]:
project.display.plotter.plot_meas_vs_calc(expt_name='d20', x_min=42, x_max=52, show_residual=True)

## Summary

This final section shows how to review the results of the analysis.

#### Show Project Summary

In [30]:
project.summary.show_report()

————————————
PROJECT INFO
————————————


Title


Untitled Project


—————————————————————
CRYSTALLOGRAPHIC DATA
—————————————————————


Phase datablock


...

lmfit (leastsq)


Fit quality


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,metric,value
1,Goodness-of-fit (reduced χ²),4.40
